# Notebook: Exhibition

[Launch on MyBinder](https://mybinder.org/v2/gh/NFDI4Culture/John-Baldessari-Exhibition-Catalogue/main?urlpath=lab/tree/notebooks)

[Notebook on GitHub](https://github.com/NFDI4Culture/John-Baldessari-Exhibition-Catalogue/blob/main/notebooks/01-exhibition.ipynb)

A Juypter Notebook for retreiving exhibition information from Wikidata item <https://www.wikidata.org/wiki/Q141747732> \| Exhibition title: while something is happening here, something else is happening there : works 1988 - 1999 (Exhibition)

Here is a list of other exhibitions from the Sprengel Museum as example to input:

    Q138547468 
    Q138572978 
    Q138572994
    Q138572988
    Q138772341
    Q138573058
    Q138650553
    Q138573096
    Q138652613
    Q138638019

The default being:

Q141747732 - while something is happening here, something else is happening there ; works 1988 - 1999

In [6]:
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json

# Target exhibition item
item_id = "Q141747732"

# Minimal property set aligned to exhibition-datamodel.qmd
property_map = {
    "P31": "Instance of",
    "P1476": "Title",
    "P276": "Location",
    "P580": "Start time",
    "P582": "End time",
    "P1640": "Curator",
    "P710": "Participant",
    "P856": "Official website",
    "P664": "Organizer",
    "P17": "Country",
    "P407": "Language",
    "P921": "Theme",
    "P859": "Sponsor",
    "P1174": "Number of visitors",
    "P373": "Commons category",
}

value_props = " ".join(f"wd:{pid}" for pid in property_map)

# Uses truthy direct claims (wdt:) for a lightweight, fast query.
query = f"""
SELECT ?prop ?propLabel ?value ?valueLabel WHERE {{
  BIND(wd:{item_id} AS ?item)
  VALUES ?prop {{ {value_props} }}

  BIND(IRI(CONCAT("http://www.wikidata.org/prop/direct/", STRAFTER(STR(?prop), "entity/"))) AS ?wdtProp)
  ?item ?wdtProp ?value .

  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en,[AUTO_LANGUAGE]". }}
}}
ORDER BY ?prop ?valueLabel
"""

endpoint = "https://query.wikidata.org/sparql"
params = urlencode({"query": query, "format": "json"})
url = f"{endpoint}?{params}"

request = Request(
    url,
    headers={
        "Accept": "application/sparql-results+json",
        "User-Agent": "loe-posit-notebook/1.0 (contact: simon.worthington@tib.eu)",
    },
)

with urlopen(request, timeout=60) as response:
    data = json.loads(response.read().decode("utf-8"))

rows = []
for b in data["results"]["bindings"]:
    prop_uri = b["prop"]["value"]
    pid = prop_uri.rsplit("/", 1)[-1]
    prop_name = property_map.get(pid, b.get("propLabel", {}).get("value", pid))
    value_text = b.get("valueLabel", {}).get("value", b["value"]["value"])
    rows.append(
        {
            "property_id": pid,
            "property": prop_name,
            "value": value_text,
        }
    )

# De-duplicate and sort for stable output
rows = list({(r["property_id"], r["value"]): r for r in rows}.values())
rows.sort(key=lambda r: (r["property_id"], r["value"].lower()))

# Print as a compact text table (no extra dependencies required)
headers = ["property_id", "property", "value"]
widths = {h: len(h) for h in headers}
for r in rows:
    for h in headers:
        widths[h] = max(widths[h], len(str(r[h])))

line = " | ".join(h.ljust(widths[h]) for h in headers)
sep = "-+-".join("-" * widths[h] for h in headers)
print(f"Wikidata item: {item_id}")
print(line)
print(sep)
for r in rows:
    print(" | ".join(str(r[h]).ljust(widths[h]) for h in headers))

print(f"\nTotal rows: {len(rows)}")

Wikidata item: Q141747732
property_id | property         | value                                                                                   
------------+------------------+-----------------------------------------------------------------------------------------
P1476       | Title            | while something is happening here, something else is happening there : works 1988 - 1999
P17         | Country          | Germany                                                                                 
P276        | Location         | Sprengel Museum                                                                         
P31         | Instance of      | exhibition                                                                              
P373        | Commons category | John Baldessari                                                                         
P580        | Start time       | 1999-09-26T00:00:00Z                                                                   